# FinAgent: DART·ECOS·FRED·Signal MCP 기반 Multi-Agent 의사결정 하네스
**BDAI 13기 [07반] 5주차 복습 과제** — 기존 퀀트 시스템(`trading_signal_lab`)을 Multi-Agent + MCP 구조로 고도화.

구조: Macro / Fundamental / Signal Agent → **Risk Harness(G1~G7)** → Portfolio Agent → 설명 → `PENDING_HUMAN_APPROVAL`.
원칙: *LLM은 판단하지 않는다, 구조가 판단한다.* 모든 판단은 결정론적 규칙이고, 주문 도구는 존재하지 않는다.

> ⚠️ 이 노트북은 **fixture 모드**(가상 DART/ECOS/FRED 값 + 합성 OHLCV)로 실행된 결과다. 실제 시장 데이터가 아니며, 실제 API 호출은 이 환경에서 검증하지 못했다.

In [1]:
import os, sys, json, asyncio, pandas as pd
sys.path.insert(0, os.getcwd())
# FINAGENT_CORE 기본값 = 상위 폴더(리포 루트). 별도 zip 불필요.
import finagent_orchestrator as fo

## 1. MCP 서버 4종이 노출하는 도구 (실제 stdio MCP 연결)
서버마다 필요한 도구만 노출한다(인터페이스 분리).

In [2]:
async def tools():
    async with fo.MCPHub('baseline', fo.Trace()) as hub:
        return await hub.list_tools()
await tools()

{'dart': ['dart_financials_impl', 'dart_disclosures_impl'],
 'ecos': ['ecos_series_impl'],
 'fred': ['fred_series_impl'],
 'signal': ['signal_evaluate_impl']}

## 2. 시나리오 매트릭스
종목 fixture × 시장 시나리오. 각 줄이 실제 MCP 호출 → 에이전트 → Harness를 거친 결과다.

In [3]:
cases = [("FIX_LONG","baseline"),("FIX_LONG","event_risk"),("FIX_LONG","risk_off"),
         ("FIX_POS","risk_off"),("FIX_NEG","baseline"),("FIX_NONE","baseline"),("FIX_LONG","bad_data")]
results = {}
rows = []
for sym, sc in cases:
    r = await fo.run(sym, "FIX001", sc)
    results[(sym, sc)] = r
    eff = [f"{g['id']}:{g['effect']}" for g in r["gates"] if g["effect"] and g["id"] != "G7"]
    rows.append(dict(종목=sym, 시나리오=sc,
                     evaluator_원판정=(r["signal"]["raw"].get("verdict") if r["signal"]["complete"] else "N/A"),
                     최종=r["action"], 개입한_게이트=" / ".join(eff) or "-"))
pd.DataFrame(rows)

,종목,시나리오,evaluator_원판정,최종,개입한_게이트
0,FIX_LONG,baseline,매수,매수,-
1,FIX_LONG,event_risk,매수,보유,G4:신규 매수 차단 → 보유
2,FIX_LONG,risk_off,매수,보유,G6:신규 매수 차단 → 보유
3,FIX_POS,risk_off,매도,매도,-
4,FIX_NEG,baseline,보유,보유,-
5,FIX_NONE,baseline,보유,보유,-
6,FIX_LONG,bad_data,N/A,보유,G2:HOLD(데이터 불완전 → 판단 보류)


읽는 법
- `FIX_LONG/baseline`: 모든 게이트 통과 → 매수 *제안*(비중 상한 5%, 가정값).
- `event_risk`: 고위험 공시(유상증자·감사의견)·부채비율·순손실 → **G4**가 매수를 보유로 낮춤.
- `risk_off`: VIX 32·장단기 역전·원/달러 급등 → **G6**이 신규 매수 차단. 같은 상황에서 매도 신호(`FIX_POS`)는 막지 않는다(위험 축소는 허용).
- `FIX_NEG`: 아래 3절 참고. `bad_data`: 소스 하나라도 실패 → **G2**가 fail-closed로 보유.

## 3. 핵심 발견 — 기존 evaluator의 부호 무시(sign-blind) 검증 (→ main에서 수정 완료)
과제 제출 시점의 `advisor/evaluator.py`는 `validated = abs(t_stat) >= t_threshold`로 검증했다. 수익률은 이미 방향(`direction`)을 곱한 값이라 **t < 0 은 "유의하게 손실 나는 신호"** 인데도 '검증됨'으로 분류되어 매수/매도 다수결에 들어갔다. 당시에는 저장소를 수정하지 않고 Harness **G5**가 `t ≥ +3.0`으로 재검증했다.

이후 `decision/gate.py`(단일 게이트)를 `t ≥ +임계`로 고쳐 main에 합쳤다(`_smoke_test_sign_gate.py`). 따라서 이 노트북을 현재 리포에서 실행하면 evaluator 자체가 `FIX_NEG`를 '보유'로 판정하고, G5는 이중 안전장치로 남는다.

In [4]:
r = results[("FIX_NEG","baseline")]
print(json.dumps(r["signal"]["raw"]["active_signals"], ensure_ascii=False, indent=1))
print("evaluator 판정:", r["signal"]["raw"]["verdict"], "→ Harness 최종:", r["action"])

[
 {
  "label": "VP박스-POC지지",
  "direction": 1,
  "evaluator_validated": false,
  "basis": "own",
  "n_events": 225,
  "mean_return": -0.75534,
  "t_stat": -3.232
 }
]
evaluator 판정: 보유 → Harness 최종: 보유


## 4. 상세 리포트 예시 (event_risk)

In [5]:
from IPython.display import Markdown
Markdown(results[("FIX_LONG","event_risk")]["report"])

## FIX_LONG · 시나리오 `event_risk` → **보유** (상태: PENDING_HUMAN_APPROVAL)

제안: 현 비중 유지(신규 거래 없음)

| Gate | 항목 | 통과 | 근거 | 행동 영향 |
|---|---|---|---|---|
| G1 | 도구 허용목록 | ✅ | 권한 밖 호출 차단 확인: G1: macro 에이전트는 signal.signal_evaluate_impl 호출 권한이 없습니다 | - |
| G2 | 데이터 완전성 | ✅ | 모든 소스 수신 | - |
| G3 | 데이터 출처 | ✅ | FIXTURE(가상 데이터) 사용: usdkrw, base_rate, vix, curve, dart_fin, dart_dis, signal | - |
| G4 | 이벤트 리스크(공시·재무) | ⛔ | 고위험 공시: 유상증자 결정, 감사의견 한정 관련 공시; 부채비율 2.00 > 1.5; 당기순손실 | 신규 매수 차단 → 보유 |
| G5 | 신호 부호 재검증(t≥+임계) | ✅ | evaluator 판정=매수, 재검증 판정=매수 | - |
| G6 | 매크로 레짐 | ✅ | neutral | - |
| G7 | Human-in-the-loop | ✅ | 주문 도구 없음. 결과는 '제안'이며 사람 승인 대기 | PENDING_HUMAN_APPROVAL |

호출 로그: macro→ecos.ecos_series_impl[ok], macro→ecos.ecos_series_impl[ok], fundamental→dart.dart_financials_impl[ok], fundamental→dart.dart_disclosures_impl[ok], macro→fred.fred_series_impl[ok], macro→fred.fred_series_impl[ok], signal→signal.signal_evaluate_impl[ok], macro→signal.signal_evaluate_impl[BLOCKED(G1)]

## 5. Live 모드 전환
키를 환경변수로 넣으면 서버가 자동으로 live로 바뀐다 (`DART_API_KEY`, `ECOS_API_KEY`, `FRED_API_KEY`; 종목은 6자리 KRX 코드, 기업은 DART corp_code).
```
python finagent_orchestrator.py --symbol 005930 --corp <corp_code>
```
**한계(모름 포함)**: live 호출 경로는 이 샌드박스에서 네트워크가 막혀 실행해보지 못했다. ECOS 국고채 등 추가 항목 코드는 미검증이라 넣지 않았다. 비중 상한·VIX 25 등 Harness 수치는 모두 가정값이다.